# Actividad: Aprendizaje supervisado con PySpark

En esta libreta se aplica aprendizaje supervisado sobre el dataset de Lending Club. El objetivo es construir una muestra reducida, preparar los datos, dividirlos en entrenamiento y prueba, entrenar modelos supervisados disponibles en PySpark y evaluar su desempeno.

## 1. Introduccion teorica

El aprendizaje supervisado es una familia de tecnicas de aprendizaje automatico en la que un modelo aprende una relacion entre variables predictoras y una variable objetivo conocida. A partir de ejemplos historicos etiquetados, el algoritmo ajusta una funcion que posteriormente puede clasificar nuevos registros o estimar valores numericos.

En problemas de clasificacion, la variable objetivo representa una clase. En problemas de regresion, representa un valor continuo. Para esta actividad se trabaja un problema de clasificacion binaria: predecir si un prestamo termina pagado (`Fully Paid`) o incumplido/castigado (`Charged Off`).

Algoritmos representativos de aprendizaje supervisado:

- Regresion logistica.
- Arboles de decision.
- Random Forest.
- Gradient-Boosted Trees.
- Multilayer Perceptron.
- Naive Bayes.

PySpark MLlib ofrece implementaciones mediante `pyspark.ml.classification`, incluyendo `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier` y `NaiveBayes`. Tambien proporciona transformadores para preparar datos, como `StringIndexer`, `OneHotEncoder`, `Imputer`, `VectorAssembler` y `Pipeline`.

## 2. Configuracion del ambiente

Esta libreta esta pensada para ejecutarse con el ambiente de Anaconda `env-pyspark`. Si se abre desde Anaconda Navigator o Jupyter, selecciona ese kernel. Si no aparece, se puede registrar desde terminal con:

```bash
conda run -n env-pyspark python -m ipykernel install --user --name env-pyspark --display-name "Python (env-pyspark)"
```

In [1]:
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    GBTClassifier,
    LinearSVC,
    LogisticRegression,
    MultilayerPerceptronClassifier,
    RandomForestClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler

SEED = 42
TARGET_SAMPLE_SIZE = 30000

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ActividadAprendizajeSupervisadoLendingClub")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

print("Spark iniciado:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/24 21:02:16 WARN Utils: Your hostname, MacBook-Pro-de-DR.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.78 instead (on interface en0)
26/05/24 21:02:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 21:02:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark iniciado: 4.1.1


## 3. Seleccion de datos

La base de datos corresponde a Lending Club, con informacion historica de prestamos y variables relacionadas con el perfil del deudor, caracteristicas del prestamo y estado final del credito.

Como punto de partida se toman los estratos definidos en la etapa previa:

- `grade_group`: grupo de riesgo derivado de `grade`.
- `home_ownership_group`: regimen de vivienda derivado de `home_ownership`.
- `stratum`: combinacion de ambas variables.

Para acotar el procesamiento individual se construye una muestra reducida `M'` mediante muestreo estratificado proporcional sobre `stratum`.

In [2]:
# Columnas usadas por el modelo.
NUMERIC_COLUMNS = [
    "loan_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "total_acc",
]

CATEGORICAL_COLUMNS = [
    "term",
    "grade",
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
]

RAW_COLUMNS = ["loan_status", *NUMERIC_COLUMNS, *CATEGORICAL_COLUMNS]

# Descarga directa de la base de datos Lending Club usando KaggleHub.
# El dataset se guarda y se busca en la carpeta del proyecto.
import os
import sys
import glob
import shutil
import subprocess
from pathlib import Path

DATASET_ID = "wordsforthewise/lending-club"
TARGET_FILE = "accepted_2007_to_2018q4.csv"
DATA_DIR = Path("/Users/SINUHE/Documents/BIg data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CSV = DATA_DIR / TARGET_FILE


def ensure_kagglehub():
    try:
        import kagglehub
        return kagglehub
    except ModuleNotFoundError:
        print("kagglehub no esta instalado. Instalando en el kernel actual...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
        import kagglehub
        return kagglehub


def find_accepted_csv(search_roots):
    patterns = [TARGET_FILE, "*accepted*2007*2018*.csv", "*accepted*.csv"]
    for root in search_roots:
        root = Path(root).expanduser()
        if not root.exists():
            continue
        for pattern in patterns:
            matches = sorted(root.rglob(pattern))
            matches = [m for m in matches if m.is_file()]
            if matches:
                return matches[0]
    return None


# 1) Primero busca el archivo ya guardado en la carpeta del proyecto.
csv_path = find_accepted_csv([DATA_DIR])

# 2) Si no existe, descarga desde KaggleHub y copia el CSV a DATA_DIR.
if csv_path is None:
    kagglehub = ensure_kagglehub()
    dataset_path = Path(kagglehub.dataset_download(DATASET_ID))
    print("Dataset descargado en cache:", dataset_path)

    downloaded_csv = find_accepted_csv([dataset_path])
    if downloaded_csv is None:
        raise FileNotFoundError(
            "No se encontro el archivo accepted_2007_to_2018q4.csv despues de la descarga."
        )

    if downloaded_csv.resolve() != LOCAL_CSV.resolve():
        print("Copiando CSV a:", LOCAL_CSV)
        shutil.copy2(downloaded_csv, LOCAL_CSV)

    csv_path = LOCAL_CSV

csv_file = str(csv_path)
print("Archivo utilizado:", csv_file)
print("Columnas a leer:", len(RAW_COLUMNS))

Archivo utilizado: /Users/SINUHE/Documents/BIg data/accepted_2007_to_2018q4.csv
Columnas a leer: 18


In [3]:
df_raw = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(csv_file)
    .select(*RAW_COLUMNS)
)

print("Columnas seleccionadas:", len(df_raw.columns))
df_raw.printSchema()

Columnas seleccionadas: 18
root
 |-- loan_status: string (nullable = true)
 |-- loan_amnt: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- dti: string (nullable = true)
 |-- fico_range_low: string (nullable = true)
 |-- fico_range_high: string (nullable = true)
 |-- open_acc: string (nullable = true)
 |-- pub_rec: string (nullable = true)
 |-- revol_bal: string (nullable = true)
 |-- total_acc: string (nullable = true)
 |-- term: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- application_type: string (nullable = true)



## 4. Preparacion de la variable objetivo

La variable objetivo seleccionada es `loan_status`, porque representa el resultado del credito. Se transforma en una etiqueta binaria:

- `Fully Paid` -> `label = 0`
- `Charged Off` -> `label = 1`

Se excluyen otros estados porque no representan un resultado final comparable para este experimento supervisado.

In [4]:
def numeric_cast(column_name):
    # Spark 4 usa modo ANSI: cast("") a double falla. try_cast tolera vacios y texto mal formado.
    cleaned = F.regexp_replace(F.col(column_name).cast("string"), "[^0-9.\\-]", "")
    cleaned = F.when(F.length(F.trim(cleaned)) == 0, None).otherwise(cleaned)
    return cleaned.try_cast("double")


df_prepared = df_raw
for column_name in NUMERIC_COLUMNS:
    df_prepared = df_prepared.withColumn(column_name, numeric_cast(column_name))

df_prepared = (
    df_prepared
    .withColumn(
        "label",
        F.when(F.col("loan_status") == "Charged Off", F.lit(1.0))
        .when(F.col("loan_status") == "Fully Paid", F.lit(0.0))
        .otherwise(F.lit(None)),
    )
    .withColumn(
        "grade_group",
        F.when(F.col("grade").isin("A", "B"), "High_Grade")
        .when(F.col("grade").isin("C", "D"), "Medium_Grade")
        .when(F.col("grade").isin("E", "F", "G"), "Low_Grade")
        .otherwise("Unknown"),
    )
    .withColumn(
        "home_ownership_group",
        F.when(F.col("home_ownership") == "MORTGAGE", "MORTGAGE")
        .when(F.col("home_ownership") == "RENT", "RENT")
        .when(F.col("home_ownership") == "OWN", "OWN")
        .otherwise("Other"),
    )
    .withColumn("stratum", F.concat_ws("_", F.col("grade_group"), F.col("home_ownership_group")))
    .dropna(subset=["label", "stratum"])
    .filter((F.col("grade_group") != "Unknown") & (F.col("home_ownership_group") != "Other"))
)

for column_name in CATEGORICAL_COLUMNS:
    df_prepared = df_prepared.withColumn(
        column_name,
        F.when(F.col(column_name).isNull(), "Unknown").otherwise(F.col(column_name)),
    )

df_prepared = df_prepared.cache()

print("Registros utiles para clasificacion:", df_prepared.count())
df_prepared.groupBy("loan_status", "label").count().orderBy("label").show(truncate=False)

Registros utiles para clasificacion: 1344831
+-----------+-----+-------+
|loan_status|label|count  |
+-----------+-----+-------+
|Fully Paid |0.0  |1076363|
|Charged Off|1.0  |268468 |
+-----------+-----+-------+



## 5. Construccion de la muestra M'

Para reducir el costo de procesamiento se construye una muestra `M'` de tamano contenido. Se usa muestreo estratificado proporcional sobre `stratum`, con lo cual se conserva la estructura de riesgo/vivienda definida en la etapa previa.

In [5]:
total_count = df_prepared.count()
sample_fraction = min(1.0, TARGET_SAMPLE_SIZE / total_count)
sample_fractions = {
    row["stratum"]: sample_fraction
    for row in df_prepared.select("stratum").distinct().collect()
}

df_sample = df_prepared.sampleBy("stratum", fractions=sample_fractions, seed=SEED).cache()

print("Tamano de M':", df_sample.count())
df_sample.groupBy("stratum").count().orderBy("stratum").show(truncate=False)

Tamano de M': 30055
+---------------------+-----+
|stratum              |count|
+---------------------+-----+
|High_Grade_MORTGAGE  |7429 |
|High_Grade_OWN       |1477 |
|High_Grade_RENT      |5152 |
|Low_Grade_MORTGAGE   |1350 |
|Low_Grade_OWN        |326  |
|Low_Grade_RENT       |1285 |
|Medium_Grade_MORTGAGE|6134 |
|Medium_Grade_OWN     |1396 |
|Medium_Grade_RENT    |5506 |
+---------------------+-----+



## 6. Division entrenamiento/prueba

La division se realiza con proporcion 70/30: 70% para entrenamiento y 30% para prueba. Se usa muestreo estratificado por la combinacion de `stratum` y `label`, reduciendo el riesgo de sesgo porque se conserva la distribucion de los grupos estructurales y de la variable objetivo.

In [6]:
df_split_ready = (
    df_sample
    .withColumn("row_id", F.monotonically_increasing_id())
    .withColumn("split_stratum", F.concat_ws("_", F.col("stratum"), F.col("label").cast("int")))
    .cache()
)

test_fractions = {
    row["split_stratum"]: 0.30
    for row in df_split_ready.select("split_stratum").distinct().collect()
}

test_data = df_split_ready.sampleBy("split_stratum", fractions=test_fractions, seed=SEED)
train_data = df_split_ready.join(test_data.select("row_id"), on="row_id", how="left_anti")

train_data = train_data.drop("row_id", "split_stratum")
test_data = test_data.drop("row_id", "split_stratum")

print("Entrenamiento:", train_data.count())
print("Prueba:", test_data.count())

print("Distribucion train:")
train_data.groupBy("label").count().orderBy("label").show()

print("Distribucion test:")
test_data.groupBy("label").count().orderBy("label").show()

Entrenamiento: 21050
Prueba: 9005
Distribucion train:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|16845|
|  1.0| 4205|
+-----+-----+

Distribucion test:
+-----+-----+
|label|count|
+-----+-----+
|  0.0| 7211|
|  1.0| 1794|
+-----+-----+



## 7. Pipeline de preparacion y modelado

El pipeline aplica las siguientes etapas:

- Imputacion de numericos con mediana.
- Indexacion de categoricas con `StringIndexer`.
- Codificacion one-hot con `OneHotEncoder`.
- Ensamble de variables con `VectorAssembler`.
- Escalamiento con `StandardScaler` para apoyar modelos lineales y redes neuronales.
- Entrenamiento de distintos clasificadores supervisados.

Se separa el preprocesamiento del modelo para poder reutilizar las mismas variables transformadas en todos los algoritmos y calcular el tamano del vector requerido por `MultilayerPerceptronClassifier`.

In [7]:
def build_preprocessing_pipeline():
    imputed_columns = [f"{column_name}_imputed" for column_name in NUMERIC_COLUMNS]
    categorical_indexed = [f"{column_name}_idx" for column_name in CATEGORICAL_COLUMNS]
    categorical_encoded = [f"{column_name}_vec" for column_name in CATEGORICAL_COLUMNS]

    indexers = [
        StringIndexer(
            inputCol=column_name,
            outputCol=f"{column_name}_idx",
            handleInvalid="keep",
        )
        for column_name in CATEGORICAL_COLUMNS
    ]

    return Pipeline(
        stages=[
            Imputer(inputCols=NUMERIC_COLUMNS, outputCols=imputed_columns, strategy="median"),
            *indexers,
            OneHotEncoder(inputCols=categorical_indexed, outputCols=categorical_encoded, handleInvalid="keep"),
            VectorAssembler(inputCols=[*imputed_columns, *categorical_encoded], outputCol="features"),
            StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=False),
        ]
    )


preprocessing_model = build_preprocessing_pipeline().fit(train_data)
train_features = preprocessing_model.transform(train_data).cache()
test_features = preprocessing_model.transform(test_data).cache()

feature_size = train_features.select("scaled_features").first()["scaled_features"].size
print("Numero de variables finales despues de codificacion:", feature_size)

26/05/24 21:02:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Numero de variables finales despues de codificacion: 56


## 8. Entrenamiento de modelos

Se entrenan seis modelos supervisados disponibles en PySpark:

- `DecisionTreeClassifier`: modelo base interpretable.
- `RandomForestClassifier`: ensamble de arboles para mejorar estabilidad.
- `GBTClassifier`: ensamble boosting que corrige errores de arboles previos.
- `LogisticRegression`: modelo lineal probabilistico, util como linea base interpretable.
- `MultilayerPerceptronClassifier`: red neuronal multicapa para capturar relaciones no lineales.
- `LinearSVC`: modelo lineal de margen. 

In [8]:
models = [
    (
        "Decision Tree",
        DecisionTreeClassifier(
            labelCol="label",
            featuresCol="scaled_features",
            maxDepth=6,
            seed=SEED,
        ),
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            labelCol="label",
            featuresCol="scaled_features",
            numTrees=30,
            maxDepth=8,
            seed=SEED,
        ),
    ),
    (
        "GBTClassifier",
        GBTClassifier(
            labelCol="label",
            featuresCol="scaled_features",
            maxIter=40,
            maxDepth=5,
            seed=SEED,
        ),
    ),
    (
        "Logistic Regression",
        LogisticRegression(
            labelCol="label",
            featuresCol="scaled_features",
            maxIter=50,
            regParam=0.01,
            elasticNetParam=0.0,
        ),
    ),
    (
        "Multilayer Perceptron",
        MultilayerPerceptronClassifier(
            labelCol="label",
            featuresCol="scaled_features",
            layers=[feature_size, 32, 16, 2],
            maxIter=80,
            blockSize=128,
            seed=SEED,
        ),
    ),
    (
        "Linear SVC",
        LinearSVC(
            labelCol="label",
            featuresCol="scaled_features",
            maxIter=50,
            regParam=0.01,
        ),
    ),
]

## 9. Evaluacion

Se usan metricas generales y metricas enfocadas en la clase de mayor interes:

- AUC ROC: capacidad de separar prestamos pagados e incumplidos.
- Accuracy: proporcion total de predicciones correctas.
- F1 ponderado: balance global entre precision y exhaustividad.
- Recall de `Charged Off`: proporcion de incumplimientos detectados.
- Precision de `Charged Off`: proporcion de predicciones de incumplimiento que realmente fueron incumplimientos.

Tambien se imprime una matriz de confusion por modelo. En riesgo crediticio, recall y precision de `Charged Off` son especialmente importantes porque la clase minoritaria representa el evento de riesgo.

In [9]:
def evaluate_predictions(predictions, model_name):
    auc = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    ).evaluate(predictions)

    accuracy = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy",
    ).evaluate(predictions)

    f1 = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1",
    ).evaluate(predictions)

    confusion = {
        (row["label"], row["prediction"]): row["count"]
        for row in predictions.groupBy("label", "prediction").count().collect()
    }
    true_positive = confusion.get((1.0, 1.0), 0)
    false_negative = confusion.get((1.0, 0.0), 0)
    false_positive = confusion.get((0.0, 1.0), 0)

    charged_off_recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    charged_off_precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0

    print(f"\n=== Resultados: {model_name} ===")
    print(f"Area bajo ROC (AUC): {auc:.4f}")
    print(f"Exactitud (accuracy): {accuracy:.4f}")
    print(f"F1 ponderado: {f1:.4f}")
    print(f"Recall Charged Off: {charged_off_recall:.4f}")
    print(f"Precision Charged Off: {charged_off_precision:.4f}")

    print("\nMatriz de confusion:")
    predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

    return {
        "modelo": model_name,
        "auc": auc,
        "accuracy": accuracy,
        "f1": f1,
        "charged_off_recall": charged_off_recall,
        "charged_off_precision": charged_off_precision,
    }


results = []

for model_name, classifier in models:
    fitted_model = classifier.fit(train_features)
    predictions = fitted_model.transform(test_features)
    results.append(evaluate_predictions(predictions, model_name))


=== Resultados: Decision Tree ===
Area bajo ROC (AUC): 0.5533
Exactitud (accuracy): 0.8003
F1 ponderado: 0.7259
Recall Charged Off: 0.0401
Precision Charged Off: 0.4865

Matriz de confusion:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7135|
|  0.0|       1.0|   76|
|  1.0|       0.0| 1722|
|  1.0|       1.0|   72|
+-----+----------+-----+



26/05/24 21:02:30 WARN DAGScheduler: Broadcasting large task binary with size 1199.8 KiB



=== Resultados: Random Forest ===
Area bajo ROC (AUC): 0.7103
Exactitud (accuracy): 0.8012
F1 ponderado: 0.7177
Recall Charged Off: 0.0145
Precision Charged Off: 0.5417

Matriz de confusion:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7189|
|  0.0|       1.0|   22|
|  1.0|       0.0| 1768|
|  1.0|       1.0|   26|
+-----+----------+-----+



26/05/24 21:02:39 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS



=== Resultados: GBTClassifier ===
Area bajo ROC (AUC): 0.7117
Exactitud (accuracy): 0.8018
F1 ponderado: 0.7394
Recall Charged Off: 0.0825
Precision Charged Off: 0.5157

Matriz de confusion:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7072|
|  0.0|       1.0|  139|
|  1.0|       0.0| 1646|
|  1.0|       1.0|  148|
+-----+----------+-----+


=== Resultados: Logistic Regression ===
Area bajo ROC (AUC): 0.7147
Exactitud (accuracy): 0.8039
F1 ponderado: 0.7292
Recall Charged Off: 0.0440
Precision Charged Off: 0.6077

Matriz de confusion:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 7160|
|  0.0|       1.0|   51|
|  1.0|       0.0| 1715|
|  1.0|       1.0|   79|
+-----+----------+-----+


=== Resultados: Multilayer Perceptron ===
Area bajo ROC (AUC): 0.7103
Exactitud (accuracy): 0.8032
F1 ponderado: 0.7357
Recall Charged Off: 0.0663
Precision Charged Off: 0.5509

Matriz de confusion:
+-----+-

In [10]:
print("=== Comparacion final ===")
for result in sorted(results, key=lambda row: row["auc"], reverse=True):
    print(
        f"{result['modelo']}: AUC={result['auc']:.4f}, "
        f"accuracy={result['accuracy']:.4f}, F1={result['f1']:.4f}, "
        f"recall_CO={result['charged_off_recall']:.4f}, "
        f"precision_CO={result['charged_off_precision']:.4f}"
    )

=== Comparacion final ===
Logistic Regression: AUC=0.7147, accuracy=0.8039, F1=0.7292, recall_CO=0.0440, precision_CO=0.6077
GBTClassifier: AUC=0.7117, accuracy=0.8018, F1=0.7394, recall_CO=0.0825, precision_CO=0.5157
Multilayer Perceptron: AUC=0.7103, accuracy=0.8032, F1=0.7357, recall_CO=0.0663, precision_CO=0.5509
Random Forest: AUC=0.7103, accuracy=0.8012, F1=0.7177, recall_CO=0.0145, precision_CO=0.5417
Linear SVC: AUC=0.6321, accuracy=0.8008, F1=0.7122, recall_CO=0.0000, precision_CO=0.0000
Decision Tree: AUC=0.5533, accuracy=0.8003, F1=0.7259, recall_CO=0.0401, precision_CO=0.4865


## 10. Resultados obtenidos

El experimento se ejecuto con el archivo local:

`/Users/SINUHE/Documents/BIg data/accepted_2007_to_2018q4.csv`

La variable objetivo quedo distribuida de la siguiente manera despues de filtrar solo estados finales:

| Clase original | Label | Registros |
|---|---:|---:|
| Fully Paid | 0 | 1,076,363 |
| Charged Off | 1 | 268,468 |

La muestra reducida `M'` quedo formada por **30,055 registros**, divididos en **21,050 registros de entrenamiento** y **9,005 registros de prueba** mediante muestreo estratificado. Despues de codificar variables categoricas y escalar las caracteristicas, el vector final tuvo **56 variables**.

Metricas obtenidas en el conjunto de prueba:

| Modelo | AUC ROC | Accuracy | F1 ponderado | Recall Charged Off | Precision Charged Off |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.7147 | 0.8039 | 0.7292 | 0.0440 | 0.6077 |
| GBTClassifier | 0.7117 | 0.8018 | 0.7394 | 0.0825 | 0.5157 |
| Multilayer Perceptron | 0.7103 | 0.8032 | 0.7357 | 0.0663 | 0.5509 |
| Random Forest | 0.7103 | 0.8012 | 0.7177 | 0.0145 | 0.5417 |
| Linear SVC | 0.6321 | 0.8008 | 0.7122 | 0.0000 | 0.0000 |
| Decision Tree | 0.5533 | 0.8003 | 0.7259 | 0.0401 | 0.4865 |

### Matrices de confusion principales

**GBTClassifier** fue el mejor en recall de `Charged Off`:

| Label real | Prediccion | Registros |
|---:|---:|---:|
| 0 | 0 | 7,072 |
| 0 | 1 | 139 |
| 1 | 0 | 1,646 |
| 1 | 1 | 148 |

**Logistic Regression** tuvo el mejor AUC y la mayor precision para `Charged Off`:

| Label real | Prediccion | Registros |
|---:|---:|---:|
| 0 | 0 | 7,160 |
| 0 | 1 | 51 |
| 1 | 0 | 1,715 |
| 1 | 1 | 79 |

### Interpretacion

El mejor AUC ROC fue el de **Logistic Regression** (**0.7147**), seguido muy de cerca por **GBTClassifier** (**0.7117**), **Multilayer Perceptron** (**0.7103**) y **Random Forest** (**0.7103**). Esto indica que varios modelos logran una separacion moderada entre prestamos pagados e incumplidos.

Sin embargo, el accuracy cercano a 80% no debe interpretarse como exito completo. La base esta desbalanceada: la mayoria de los prestamos son `Fully Paid`. Por eso, un modelo puede obtener buen accuracy aunque casi nunca detecte `Charged Off`.

Desde el punto de vista del analisis de riesgo, **GBTClassifier** aporta la mejor utilidad practica inicial porque detecta mas incumplimientos: recall de `Charged Off` = **0.0825**. Aunque este recall sigue siendo bajo, supera a los demas modelos. **Logistic Regression** es valiosa para aprendizaje y retroalimentacion porque obtuvo el mejor AUC y es mas interpretable: permite entender la direccion e importancia de las variables.

**Aclaracion sobre Linear SVC:** este modelo se incluyo como referencia didactica para comparar contra un clasificador lineal de margen, no porque se esperara que fuera el mejor modelo final. El resultado confirma una limitacion importante: con esta configuracion y con clases desbalanceadas, `LinearSVC` predijo todos los casos como `Fully Paid`. Por eso su recall y precision para `Charged Off` fueron **0.0000**. Se conserva en la comparacion porque aporta retroalimentacion metodologica: muestra que un buen accuracy puede ser enganoso y que no todos los algoritmos son adecuados sin balanceo o ajuste de decision.

## 11. Propuestas para mejorar los resultados

1. **Tratar el desbalance de clases.** La clase `Charged Off` es minoritaria. Se recomienda usar sobremuestreo de incumplidos, submuestreo de pagados o pesos de clase para que los errores sobre incumplimientos tengan mayor penalizacion.

2. **Ajustar el umbral de decision.** El umbral por defecto favorece la clase mayoritaria. Para riesgo crediticio conviene bajar el umbral de prediccion de incumplimiento y elegirlo segun una matriz de costos.

3. **Optimizar por recall o F1 de `Charged Off`.** En vez de elegir solo por AUC o accuracy, se recomienda seleccionar el modelo con base en recall, precision y F1 de la clase incumplida.

4. **Probar validacion cruzada y busqueda de hiperparametros.** Para `GBTClassifier` se pueden ajustar `maxIter`, `maxDepth`, `stepSize` y `minInstancesPerNode`. Para `RandomForestClassifier`, `numTrees`, `maxDepth` y `featureSubsetStrategy`. Para regresion logistica, `regParam` y `elasticNetParam`.

5. **Agregar variables predictoras de riesgo.** Se pueden incorporar `sub_grade`, `emp_length`, `revol_util`, `delinq_2yrs`, `inq_last_6mths`, `mort_acc`, `pub_rec_bankruptcies`, `acc_open_past_24mths` y variables historicas de morosidad.

6. **Evitar fuga de informacion.** No deben incluirse variables posteriores al otorgamiento del prestamo, como pagos recibidos, recuperaciones, saldos posteriores o fecha de ultimo pago, porque inflarian artificialmente el desempeno.

7. **Aumentar el tamano de `M'`.** Con 30,055 registros el experimento es manejable. Si el equipo lo permite, aumentar a 100,000 o mas registros podria mejorar estabilidad, especialmente para redes neuronales y GBT.

8. **Comparar con balanceo dentro del pipeline.** Una segunda ronda experimental deberia entrenar los mismos modelos con datos balanceados y comparar si mejora el recall de `Charged Off` sin perder demasiada precision.

## 12. Analisis final y conclusiones

La comparacion ampliada muestra que no siempre el modelo mas complejo gana en todas las metricas. **Logistic Regression** obtuvo el mejor AUC ROC (**0.7147**), lo que la convierte en una base solida e interpretable. **GBTClassifier** obtuvo el mejor F1 ponderado (**0.7394**) y el mejor recall de `Charged Off` (**0.0825**), por lo que es el candidato mas util si el objetivo principal es detectar mas prestamos incumplidos.

**Multilayer Perceptron** tuvo un rendimiento competitivo, pero no supero claramente a los modelos mas simples. Esto es una retroalimentacion importante: las redes neuronales no garantizan mejor desempeno en datos tabulares si no se ajustan cuidadosamente y si las variables disponibles no capturan suficiente senal.

**Linear SVC** fue util como contraste metodologico y didactico, pero no debe seleccionarse como modelo final en esta version. Se incluyo para observar el comportamiento de un clasificador lineal de margen; sin embargo, predijo todos los casos como `Fully Paid`, dejando recall y precision de `Charged Off` en cero. Este resultado justifica descartarlo como candidato final y refuerza la necesidad de balanceo de clases o ajuste del criterio de decision. **Decision Tree** quedo como una linea base interpretable, aunque con menor AUC.

La conclusion principal es que el problema no esta resuelto solo con cambiar de algoritmo. El reto central es el **desbalance de clases** y la baja deteccion de `Charged Off`. La siguiente mejora debe enfocarse en balanceo, ajuste de umbral, variables de riesgo mas informativas y seleccion de metricas alineadas al costo real de aprobar un prestamo riesgoso.

In [11]:
# Ejecutar al finalizar si se desea liberar la sesion de Spark.
# spark.stop()